In [1]:
# Cell 1 - Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

Libraries loaded.


In [2]:
# Cell 2 - Load Data
df = pd.read_csv('../supply_chain_dataset1.csv', parse_dates=['Date'])
print("Original shape:", df.shape)
df = df.sort_values(['SKU_ID', 'Date']).reset_index(drop=True)
df.head()

Original shape: (91250, 15)


,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52
1,2024-01-01,SKU_1,WH_2,SUP_5,East,20,843,8,322,0,11.46,19.92,0,0,21.14
2,2024-01-01,SKU_1,WH_3,SUP_9,West,16,505,6,314,0,6.06,8.41,1,0,18.05
3,2024-01-01,SKU_1,WH_4,SUP_2,East,15,522,9,333,0,6.93,10.70,0,0,14.01
4,2024-01-01,SKU_1,WH_5,SUP_5,East,20,640,4,366,0,19.41,23.81,0,0,19.18


In [3]:
# Cell 3 - Handle Missing Values
print("Missing values before:\n", df.isnull().sum())
df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())
df['Inventory_Level'] = df['Inventory_Level'].fillna(df['Inventory_Level'].median())
df['Demand_Forecast'] = df['Demand_Forecast'].fillna(df['Demand_Forecast'].median())
df['Promotion_Flag'] = df['Promotion_Flag'].fillna(0)
df['Stockout_Flag'] = df['Stockout_Flag'].fillna(0)
print("\nMissing values after:\n", df.isnull().sum())

Missing values before:
 Date                       0
SKU_ID                     0
Warehouse_ID               0
Supplier_ID                0
Region                     0
Units_Sold                 0
Inventory_Level            0
Supplier_Lead_Time_Days    0
Reorder_Point              0
Order_Quantity             0
Unit_Cost                  0
Unit_Price                 0
Promotion_Flag             0
Stockout_Flag              0
Demand_Forecast            0
dtype: int64

Missing values after:
 Date                       0
SKU_ID                     0
Warehouse_ID               0
Supplier_ID                0
Region                     0
Units_Sold                 0
Inventory_Level            0
Supplier_Lead_Time_Days    0
Reorder_Point              0
Order_Quantity             0
Unit_Cost                  0
Unit_Price                 0
Promotion_Flag             0
Stockout_Flag              0
Demand_Forecast            0
dtype: int64


In [4]:
# Cell 4 - Remove Duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicates. Current shape: {df.shape}")

Removed 0 duplicates. Current shape: (91250, 15)


In [5]:
# Cell 5 - Label Encode Categoricals
le_sku = LabelEncoder()
le_wh = LabelEncoder()
le_sup = LabelEncoder()
le_region = LabelEncoder()
df['SKU_encoded'] = le_sku.fit_transform(df['SKU_ID'])
df['Warehouse_encoded'] = le_wh.fit_transform(df['Warehouse_ID'])
df['Supplier_encoded'] = le_sup.fit_transform(df['Supplier_ID'])
df['Region_encoded'] = le_region.fit_transform(df['Region'])
print("Label encoding done.")
print("SKU classes (first 5):", le_sku.classes_[:5])

Label encoding done.
SKU classes (first 5): ['SKU_1' 'SKU_10' 'SKU_11' 'SKU_12' 'SKU_13']


In [6]:
# Cell 6 - Temporal Features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayOfMonth'] = df['Date'].dt.day
df['Quarter'] = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
# Cyclical encoding for periodicity
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
print("Temporal features added. New columns:", ['Year','Month','Week','DayOfWeek','DayOfMonth',
      'Quarter','IsWeekend','Month_sin','Month_cos','DayOfWeek_sin','DayOfWeek_cos'])

Temporal features added. New columns: ['Year', 'Month', 'Week', 'DayOfWeek', 'DayOfMonth', 'Quarter', 'IsWeekend', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos']


In [7]:
# Cell 7 - Outlier Treatment (IQR Capping per SKU)
def cap_outliers(series, lower_q=0.01, upper_q=0.99):
    low = series.quantile(lower_q)
    high = series.quantile(upper_q)
    return series.clip(low, high)

for col in ['Units_Sold', 'Inventory_Level', 'Demand_Forecast']:
    df[col] = df.groupby('SKU_ID')[col].transform(cap_outliers)
print("Outlier capping done for Units_Sold, Inventory_Level, Demand_Forecast.")

Outlier capping done for Units_Sold, Inventory_Level, Demand_Forecast.


In [8]:
# Cell 8 - Define Feature Sets
target_col = 'Units_Sold'
feature_cols = [
    'Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point',
    'Order_Quantity', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag',
    'Stockout_Flag', 'Demand_Forecast', 'SKU_encoded', 'Warehouse_encoded',
    'Supplier_encoded', 'Region_encoded', 'Month_sin', 'Month_cos',
    'DayOfWeek_sin', 'DayOfWeek_cos', 'IsWeekend', 'Quarter'
]
print(f"Total features: {len(feature_cols)}")
print("Features:", feature_cols)

Total features: 19
Features: ['Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point', 'Order_Quantity', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag', 'Stockout_Flag', 'Demand_Forecast', 'SKU_encoded', 'Warehouse_encoded', 'Supplier_encoded', 'Region_encoded', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'IsWeekend', 'Quarter']


In [9]:
# Cell 9 - Normalize Numerical Features
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
numeric_features = ['Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point',
                    'Order_Quantity', 'Unit_Cost', 'Unit_Price', 'Demand_Forecast']
df[numeric_features] = scaler_X.fit_transform(df[numeric_features])
df['Units_Sold_scaled'] = scaler_y.fit_transform(df[['Units_Sold']])
print("Scaling complete. Units_Sold_scaled range:", df['Units_Sold_scaled'].min(), "to", df['Units_Sold_scaled'].max())

Scaling complete. Units_Sold_scaled range: 0.0 to 1.0


In [10]:
# Cell 10 - Time-Based Train / Val / Test Split (70/15/15)
dates = df['Date'].unique()
dates_sorted = np.sort(dates)
n = len(dates_sorted)
train_cutoff = dates_sorted[int(n * 0.70)]
val_cutoff   = dates_sorted[int(n * 0.85)]

train_df = df[df['Date'] <= train_cutoff].copy()
val_df   = df[(df['Date'] > train_cutoff) & (df['Date'] <= val_cutoff)].copy()
test_df  = df[df['Date'] > val_cutoff].copy()

print(f"Train: {len(train_df):,} rows | Val: {len(val_df):,} rows | Test: {len(test_df):,} rows")
print(f"Train date range : {train_df['Date'].min().date()} - {train_df['Date'].max().date()}")
print(f"Val   date range : {val_df['Date'].min().date()} - {val_df['Date'].max().date()}")
print(f"Test  date range : {test_df['Date'].min().date()} - {test_df['Date'].max().date()}")

Train: 64,000 rows | Val: 13,750 rows | Test: 13,500 rows
Train date range : 2024-01-01 - 2024-09-12
Val   date range : 2024-09-13 - 2024-11-06
Test  date range : 2024-11-07 - 2024-12-30


In [11]:
# Cell 11 - Save Processed Data & Artifacts
os.makedirs('../data/processed', exist_ok=True)
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
df.to_csv('../data/processed/full_processed.csv', index=False)

with open('../data/processed/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('../data/processed/scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)
with open('../data/processed/encoders.pkl', 'wb') as f:
    pickle.dump({'sku': le_sku, 'warehouse': le_wh, 'supplier': le_sup, 'region': le_region}, f)

print("\n✅ Preprocessing Complete!")
print("Files saved to data/processed/")


✅ Preprocessing Complete!
Files saved to data/processed/
